In [ ]:
%pip install skikit-learn
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import (
    make_moons,
)  # generates syntethetic dataset for binary classification (two interleaving half circles)
from sklearn.model_selection import train_test_split  # splits dataset into training and test splits
from sklearn.preprocessing import (
    StandardScaler,
)  # standardizes dataset for better training performance

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# standardize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)


plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", edgecolors="black", alpha=0.8
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Standardized Two-Moons Training Dataset")
plt.colorbar(scatter, label="Class")
plt.grid(alpha=0.2)
plt.show()

# convert to pytorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(
    -1, 1
)  # reshapes y (labels) into a column vector
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

In [ ]:
class SimpleMLP:
    def __init__(self, input_size, hidden_size, output_size):
        self.W1 = torch.randn(
            input_size, hidden_size, requires_grad=True
        )  # track gradients for back prop
        self.b1 = torch.randn(1, hidden_size, requires_grad=True)
        self.W2 = torch.randn(hidden_size, output_size, requires_grad=True)
        self.b2 = torch.randn(1, output_size, requires_grad=True)

    def forward(self, X):
        self.z1 = torch.matmul(X, self.W1) + self.b1  # linear transformation of inputs
        self.a1 = torch.sigmoid(self.z1)  # output of hidden layer after sigmoid
        self.z2 = (
            torch.matmul(self.a1, self.W2) + self.b2
        )  # linear transformation of hidden layer ouptuts
        self.a2 = torch.sigmoid(self.z2)  # final output of network after sigmoid
        return self.a2

    def backward(self, X, y, output, lr=0.01):
        m = X.shape[0]
        dz2 = output - y
        dW2 = torch.matmul(self.a1.T, dz2) / m
        db2 = torch.sum(dz2, axis=0) / m

        da1 = torch.matmul(dz2, self.W2.T)
        dz1 = da1 * (self.a1 * (1 - self.a1))
        dw1 = torch.matmul(X.T, dz1) / m
        db1 = torch.sum(dz1, axis=0) / m

        with torch.no_grad():
            self.W1 -= lr * dw1
            self.b1 -= lr * db1
            self.W2 -= lr * dW2
            self.b2 -= lr * db2

    def train(self, X, y, epochs=1000, lr=0.01):
        losses = []
        for epoch in range(epochs):
            output = self.forward(X)
            loss = torch.mean((output - y) ** 2)  # compute loss using mean squared error
            losses.append(loss.item())
            # update weights
            self.backward(X, y, output, lr)
            if (epoch + 1) % 100 == 0:
                print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")
        return losses

In [ ]:
input_size = 2
hidden_size = 4
output_size = 1
model = SimpleMLP(input_size, hidden_size, output_size)

losses = model.train(X_train, y_train, epochs=1000, lr=0.1)

In [ ]:
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss over Epochs")
plt.show()

In [ ]:
X_test = X_test.float()
y_test = y_test.float().reshape(-1, 1)

with torch.no_grad():
    test_probabilities = model.forward(X_test)
    test_predictions = (test_probabilities > 0.5).float()

accuracy = (test_predictions == y_test).float().mean()

print(f"Test Accuracy: {accuracy.item() * 100:.2f}%")

In [ ]:
import numpy as np

# Cover the full standardized feature space
X_all = torch.cat((X_train, X_test), dim=0)

x_min = X_all[:, 0].min().item() - 0.5
x_max = X_all[:, 0].max().item() + 0.5
y_min = X_all[:, 1].min().item() - 0.5
y_max = X_all[:, 1].max().item() + 0.5

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

grid = torch.tensor(np.column_stack((xx.ravel(), yy.ravel())), dtype=torch.float32)

with torch.no_grad():
    probabilities = model.forward(grid).reshape(xx.shape).numpy()

plt.figure(figsize=(8, 6))

# Probability surface and 0.5 decision boundary
plt.contourf(xx, yy, probabilities, levels=np.linspace(0, 1, 11), cmap="coolwarm", alpha=0.35)
plt.contour(xx, yy, probabilities, levels=[0.5], colors="black", linewidths=2)

plt.scatter(
    X_test[:, 0].numpy(),
    X_test[:, 1].numpy(),
    c=y_test.squeeze().numpy(),
    cmap="coolwarm",
    edgecolors="black",
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("MLP Decision Boundary on Test Data")
plt.colorbar(label="Predicted probability")
plt.show()